# Global PM2.5 Analysis: From Raster to Country/Continent Means

This notebook provides the explanation and code for processing global raster air quality datasets into vector layers to efficiently save the data in a easily ingestable format for a an interactive web map.

There are three main phases to this notebook:

1. **Prequsites and loading libraries**
2. **Raster to vector conversion** - Aggregating high-resolution GeoTIFF rasters into a manageable H3 hex grid.
3. **Spatial analysis and regional aggregation** - Mapping hex values into politcial boundaries to derive regional trends.
4. **Data export for visualisation**

## 1. Prequsities and loading libraries

Due to the file sizes and computational power to download multiple rasters datasets, these must be downloaded manually before running this notebook. This provides flexibility in the choice and amount of raster files to process and use for analysis.

This notebook uses high resolution geotiff rasters from NASA EarthData - Global Annual PM2.5 Grids from MODIS, MISR, SeaWiFS and VIIRS Aerosol Optical Depth (AOD), 1998-2022, V5.GL.04. These can be downloaded manually from https://cmr.earthdata.nasa.gov/search/concepts/C3540908987-ESDIS.html.

In this instance the full 25 years (1998 - 2022) of raster data was downloaded to provide a full story of temporal air quality changes. Due to the file sizes of the geotiffs this is the most time consuming part of this analysis.

The only other dataset to predownload is a country boundary datasets, in this instance the country dataset was gathered from GeoBoundaries - https://www.geoboundaries.org/globalDownloads.html.

In [ ]:
# install required libraries
# uncomment the following lines to install the required libraries (only if required)

# !pip install rasterio h3 geopandas shapely numpy tqdm pycountry-convert

In [ ]:
# load libraries
import rasterio
from rasterio.transform import xy
import h3
import geopandas as gpd
import pandas as pd
import numpy as np
import json
import os
from shapely.geometry import Polygon, Point
from tqdm import tqdm
from pycountry_convert import country_alpha3_to_country_alpha2, country_alpha2_to_continent_code

## 2. Phase 2: Raster to vector conversion

This section focuses on making the huge raster datasets manageable for an interactive map visualisation. It details the steps for processing the raw annual PM2.5 raster files, aggregating the billions of pixels into H3 resolution 3 hexagons to normalise the data for global visualisation. This reduces the file sizes from ~300mb per raster into one single GeoJSON of ~11mb for this H3 resolution.

### 2.1 Config and Paths

Firstly, we define the range of years, the resolutions of the H3 grid and source and destination paths. 

In [ ]:
# Configuration
# Base directory where annual raster files are located
base_dir = r".\data\raster"
# Generate dictionary for years 1998 through 2022
TIFF_FILES = {
    year: os.path.join(
        base_dir, 
        f"global-annual-gwr-pm2-5-v5-gl-04-{year}-geotiff.tif"
    )
    for year in range(1998, 2023)
}
H3_RESOLUTION = 3
YEARS = list(range(1998, 2023))
DOWNSAMPLE_FACTOR = 10
OUTPUT_GEOJSON = r".\data\pm25_hex_grid_1998-2022.geojson"
NODATA_THRESHOLD = 0.0

### 2.2 Processing and aggregation

This logic iterates through each year, resamples the raster points and assigns a H3 cell. It then calculates the PM2.5 value per hexagon.

Firstly, we define functions to efficiently process the data:

1. Load rasters using downsampling for more efficient processing
2. Aggregate to H3 grid
3. Build a GeoJSON with an Antimeridian check to prevent horizontal lines across the map.
4. Define __main__ to run the aforementioned functions in sequence

In [ ]:
def load_raster_samples(tif_path, downsample_factor=10):
    """
    Open a GeoTIFF and return a list of (lon, lat, value) tuples,
    downsampled to reduce processing time.
    """
    print(f"  Loading raster: {tif_path}")
    samples = []

    with rasterio.open(tif_path) as src:
        nodata = src.nodata

        # Read at reduced resolution
        out_shape = (
            1,
            max(1, src.height // downsample_factor),
            max(1, src.width  // downsample_factor),
        )
        data = src.read(1, out_shape=out_shape, resampling=rasterio.enums.Resampling.average)

        # Compute pixel centre coordinates in the downsampled grid
        row_scale = src.height / out_shape[1]
        col_scale = src.width  / out_shape[2]

        rows, cols = np.where(data > NODATA_THRESHOLD)
        if nodata is not None:
            valid = data[rows, cols] != nodata
            rows, cols = rows[valid], cols[valid]

        # Convert pixel indices back to original-space pixel centres, then to geographic coords
        orig_rows = (rows + 0.5) * row_scale
        orig_cols = (cols + 0.5) * col_scale
        xs, ys = xy(src.transform, orig_rows, orig_cols)

        values = data[rows, cols].astype(float)
        samples = list(zip(ys, xs, values))  # (lat, lon, value)

    print(f"  → {len(samples):,} valid sample points loaded")
    return samples


def aggregate_to_h3(samples, resolution):
    """
    Aggregate (lat, lon, value) samples into H3 hexagons at the given resolution.
    Returns dict: {h3_index: [list_of_values]}
    """
    print(f"  Aggregating to H3 resolution {resolution}...")
    hex_values = {}
    for lat, lon, val in tqdm(samples, desc="  H3 assignment"):
        h = h3.latlng_to_cell(lat, lon, resolution)
        hex_values.setdefault(h, []).append(val)
    return hex_values


def build_geojson(hex_year_data, years):
    """
    Build a GeoJSON FeatureCollection with an Antimeridian check to prevent 
    horizontal lines across the map.
    """
    WHO_GUIDELINE = 5.0
    features = []
    # print(f"   Building GeoJSON features for {len(hex_year_data):,} hexagons...")

    for h_idx, year_vals in tqdm(hex_year_data.items(), desc="   Building features"):
        boundary = h3.cell_to_boundary(h_idx)
        
        # Convert (lat, lon) to [lon, lat] and fix Antimeridian crossing
        raw_coords = [[lon, lat] for lat, lon in boundary]
        
        # --- ANTIMERIDIAN FIX ---
        # If the difference between any two consecutive longitudes is > 180, 
        # the hexagon crosses the date line and creates a horizontal stripe 
        # across the map - this fixes this issue.
        lons = [c[0] for c in raw_coords]
        if max(lons) - min(lons) > 180:
            # Shift the negative coordinates to be > 180 or positive to be < -180
            # to keep the polygon "local" to one side for the GeoJSON record.
            adjusted_coords = []
            for lon, lat in raw_coords:
                if lon < 0:
                    adjusted_coords.append([lon + 360, lat])
                else:
                    adjusted_coords.append([lon, lat])
            coords = adjusted_coords
        else:
            coords = raw_coords
            
        coords.append(coords[0]) # Close the loop

        props = {"h3_index": h_idx}
        for year in years:
            val = year_vals.get(year)
            if val is not None:
                props[f"pm25_{year}"] = round(val, 2)

        latest_year = max(year_vals.keys())
        latest_val = year_vals[latest_year]
        props.update({
            "pm25_latest": round(latest_val, 2),
            "latest_year": latest_year,
            "who_exceedance": round(latest_val / WHO_GUIDELINE, 2),
            "who_exceeded": bool(latest_val > WHO_GUIDELINE)
        })

        # Simplified band logic
        bands = [(5, "Good"), (12, "Moderate"), (35, "Sensitive"), (55, "Unhealthy"), (150, "Very Unhealthy")]
        props["band"] = next((b[1] for b in bands if latest_val <= b[0]), "Hazardous")

        features.append({
            "type": "Feature",
            "geometry": {"type": "Polygon", "coordinates": [coords]},
            "properties": props,
        })

    return {"type": "FeatureCollection", "features": features}

We now define main to run the functions above, this is the longest part of this process but should complete in 7 - 10 minutes.

In [ ]:
def main():
    global hex_grid
    years = sorted(TIFF_FILES.keys())

    # --- Step 1: Load samples per year ---
    year_samples = {}
    for year, path in TIFF_FILES.items():
        print(f"[Year {year}]")
        year_samples[year] = load_raster_samples(path, DOWNSAMPLE_FACTOR)
        print()

    # --- Step 2: Aggregate each year's samples into H3 hexes ---
    # Master dict: {h3_index: {year: mean_value}}
    hex_year_data = {}

    for year, samples in year_samples.items():
        print(f"[Year {year}] Aggregating to H3...")
        hex_vals = aggregate_to_h3(samples, H3_RESOLUTION)
        for h_idx, vals in hex_vals.items():
            if h_idx not in hex_year_data:
                hex_year_data[h_idx] = {}
            hex_year_data[h_idx][year] = float(np.mean(vals))

    # --- Step 3: Build GeoJSON ---
    print("[Building GeoJSON]")
    geojson = build_geojson(hex_year_data, years)

    # --- Step 4: Write output ---
    print(f"\n[Writing] {OUTPUT_GEOJSON}")
    with open(OUTPUT_GEOJSON, "w") as f:
        json.dump(geojson, f, separators=(",", ":"))
        
    size_mb = __import__("os").path.getsize(OUTPUT_GEOJSON) / 1e6
    
    # --- Step 5: Convert to GeoDataFrame for further analysis ---
    print("[Creating GeoDataFrame]")
    # Convert the GeoJSON features into a GeoDataFrame and set crs
    hex_grid = gpd.GeoDataFrame.from_features(geojson['features'])
    hex_grid.set_crs("EPSG:4326", inplace=True)

    print(f"  → hex_grid ready with {len(hex_grid):,} hexagons.")

if __name__ == "__main__":
    main()

## 3. Phase 3: Spatial analysis and regional aggregation

With the create hex grid we can now overlay this with country boundaries to determine which hexagon relates to which country and continent.

### 3.1 Load datasets and project

In [ ]:
# Load Country Boundaries and hex grid
countries = gpd.read_file("data/country_data.geojson")
hex_grid = gpd.read_file("data/pm25_hex_grid_1998-2022.geojson")

# Ensure CRS consistency by setting CRS
if countries.crs != hex_grid.crs:
    hex_grid = hex_grid.to_crs(countries.crs)

### 3.2 Continent Mapping

Using *pycountry-convert* to map ISO Alpha 3 country codes and the respective continents, by first creating a function get_continent_name and applying to countries dataset.

In [ ]:
def get_continent_name(iso_alpha3):
    continent_map = {
        'AF': 'Africa', 'AS': 'Asia', 'EU': 'Europe', 
        'NA': 'North America', 'SA': 'South America', 
        'OC': 'Oceania', 'AN':'Antarctica'
    }
    try:
        alpha2 = country_alpha3_to_country_alpha2(iso_alpha3)
        cont_code = country_alpha2_to_continent_code(alpha2)
        return continent_map.get(cont_code, 'Unknown')
    except:
        return 'Unknown'
    
# Apply continent mapping to countries
countries['continent'] = countries['C_abb'].apply(get_continent_name)

### 3.2 Spatial join and grouping

Now perform a spatial oin using the centroids of each hexagone to ensure that each hexagon is assigned to only one country based on the centre point.

This uses the previously downloaded country_data.geojson from GeoBoundaries - https://www.geoboundaries.org/globalDownloads.html - but the source can be amended as required.

In [ ]:
# Spatial Join - Project to EPSG: 3857 to calculate 
# centroids in metres accurately, then spatially join
hex_centroids = hex_grid.copy()
hex_centroids['geometry'] = hex_grid.geometry.to_crs("EPSG:3857").centroid.to_crs(hex_grid.crs)

# Identify which country each hex centre belongs to - adding C_name here
joined_data = gpd.sjoin(
    hex_centroids, 
    countries[['C_abb', 'C_name', 'continent', 'geometry']], 
    how="inner", 
    predicate="within"
)

### 3.4 Calculate annual means by country and by continent

We can now calculate country and continental means which can be used for baseline stats in the final charts.

In [ ]:
# Calculate Annual Means
year_cols = [f"pm25_{y}" for y in YEARS]
country_avgs = joined_data.groupby('C_abb')[year_cols].mean()
continent_avgs = joined_data.groupby('continent')[year_cols].mean()

## 4. Data export for visualisation

Finally, the results are formatted into GeoJSON and JSON structure suitable for our visualisations (map and charts). This contains both the country and continent means for each year processed to provide benchmarks for the final visualisations.

### 4.1 Export GeoJSON of hex grids for mapping

In [ ]:
final_hex_grid = hex_grid.merge(
    joined_data[['C_abb', 'C_name', 'continent']], 
    left_index=True, 
    right_index=True, 
    how='inner'
)
final_hex_grid.to_file("./data/pm25_hex_grid_processed.geojson", driver='GeoJSON')

print("Processing Complete. File Saved: Hex Grid GeoJSON")

### 4.2 Export JSON of mean values for charts

In [ ]:
# Format JSON for charts
chart_json = {
    "countries": country_avgs.transpose().to_dict(orient='list'),
    "continents": continent_avgs.transpose().to_dict(orient='list'),
    "mapping": countries.set_index('C_abb')['continent'].to_dict(),
    "names": countries.set_index('C_abb')['C_name'].to_dict()
}

with open('./data/pm25_averages.json', 'w') as f:
    json.dump(chart_json, f)

print("Processing Complete. File saved: PM2.5 averages JSON")